# Impacts of Over Sampling

### Adasyn

In [1]:
import pickle
import numpy as np
import os
import time

from imblearn.over_sampling import ADASYN


def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]


# ---------------------------------------------------------
# Load saved ENN-cleaned dataset
# ---------------------------------------------------------
folder = "./final_split_data_HybridNorm_ENN"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

print("Loaded ENN-cleaned dataset:")
print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape,   y_val.shape)
print("Test :", X_test.shape,  y_test.shape)


# ---------------------------------------------------------
# Prepare indices
# ---------------------------------------------------------
N, T, F = X_train.shape

y_train = y_train.astype(int)

nonsep_idx = np.where(y_train == 0)[0]
sep_idx    = np.where(y_train == 1)[0]

original_nonsep = len(nonsep_idx)
original_sep    = len(sep_idx)

X_real_sep_all = X_train[sep_idx].astype(np.float32)
X_nonsep_all   = X_train[nonsep_idx].astype(np.float32)

print("\nOriginal ENN-cleaned training class counts:")
print("Non-SEP:", original_nonsep)
print("SEP    :", original_sep)
print(f"Ratio  : {original_nonsep / original_sep:.1f}x")
print("─" * 80)


# ---------------------------------------------------------
# Create synthetic SEP samples once using ADASYN
# Generate enough synthetic SEP for the largest target
# ---------------------------------------------------------
rng = np.random.default_rng(seed=42)

N_SYNTHETIC_SEP = max(12000, original_nonsep - original_sep)

print(f"\nSynthetic SEP samples to generate: {N_SYNTHETIC_SEP}")

X_train_flat = X_train.reshape(X_train.shape[0], T * F)

n_neighbors = min(3, original_sep - 1)

if n_neighbors < 1:
    raise ValueError(
        f"Not enough SEP samples for ADASYN. SEP count = {original_sep}"
    )

adasyn = ADASYN(
    sampling_strategy={1: original_sep + N_SYNTHETIC_SEP},
    n_neighbors=n_neighbors,
    random_state=42
)

# ---------------------------------------------------------
# Measure ADASYN synthetic generation time
# ---------------------------------------------------------
adasyn_start_time = time.time()

X_res_flat, y_res = adasyn.fit_resample(X_train_flat, y_train)

adasyn_infer_time = time.time() - adasyn_start_time

# ADASYN appends synthetic samples after the original samples
n_original = len(X_train_flat)

X_synthetic_flat = X_res_flat[n_original:]
y_synthetic      = y_res[n_original:]

X_synthetic_sep_all = X_synthetic_flat[y_synthetic.astype(int) == 1]
X_synthetic_sep_all = X_synthetic_sep_all.reshape(-1, T, F).astype(np.float32)

n_synthetic_generated = len(X_synthetic_sep_all)

print("\nCreated ADASYN synthetic SEP pool:")
print(f"Synthetic SEP shape           : {X_synthetic_sep_all.shape}")
print(f"Synthetic SEP count           : {n_synthetic_generated}")
print(f"ADASYN generation time        : {adasyn_infer_time:.4f}s")
print(f"ADASYN time per synthetic SEP : {adasyn_infer_time / n_synthetic_generated * 1000:.4f}ms")
print("─" * 80)


# ---------------------------------------------------------
# Save real vs synthetic SEP samples once
# ---------------------------------------------------------
SAVE_DIR = "./sep_samples"
os.makedirs(SAVE_DIR, exist_ok=True)

n_compare = min(118, len(X_real_sep_all), len(X_synthetic_sep_all))

idx_real_compare = rng.choice(
    len(X_real_sep_all),
    size=n_compare,
    replace=False
)

idx_synthetic_compare = rng.choice(
    len(X_synthetic_sep_all),
    size=n_compare,
    replace=False
)

save_path = os.path.join(SAVE_DIR, "sep_real_vs_synthetic_adasyn.pkl")

with open(save_path, "wb") as f:
    pickle.dump({
        "X_real_sep": X_real_sep_all[idx_real_compare],
        "y_real_sep": np.zeros(n_compare, dtype=np.int32),      # 0 = real

        "X_synthetic_sep": X_synthetic_sep_all[idx_synthetic_compare],
        "y_synthetic_sep": np.ones(n_compare, dtype=np.int32),  # 1 = synthetic

        "n_real_sep_total": original_sep,
        "n_synthetic_sep_total": len(X_synthetic_sep_all),
        "n_compare": n_compare,
        "n_neighbors": n_neighbors,
        "adasyn_infer_time": adasyn_infer_time,
        "adasyn_time_per_sample_ms": adasyn_infer_time / n_synthetic_generated * 1000,
    }, f)

print(f"Saved real vs synthetic ADASYN SEP samples → {os.path.abspath(save_path)}")
print(f"Real SEP      : {n_compare}")
print(f"Synthetic SEP : {n_compare}")
print("─" * 80)


# ---------------------------------------------------------
# Dataset configurations
# ---------------------------------------------------------
# enn_adasyn_balanced:
#   No RUS. Keep all Non-SEP.
#   Use synthetic SEP to make total SEP = total Non-SEP.
#
# enn_rus*_adasyn*:
#   RUS Non-SEP to target.
#   Use synthetic SEP to make total SEP = target.

adasyn_configs = {
    "enn_adasyn_balanced": {
        "nonsep_target": original_nonsep,
        "sep_target": original_nonsep,
        "use_rus": False,
    },

    "enn_rus8000_adasyn8000": {
        "nonsep_target": 8000,
        "sep_target": 8000,
        "use_rus": True,
    },

    "enn_rus4000_adasyn4000": {
        "nonsep_target": 4000,
        "sep_target": 4000,
        "use_rus": True,
    },

    "enn_rus2000_adasyn2000": {
        "nonsep_target": 2000,
        "sep_target": 2000,
        "use_rus": True,
    },

    "enn_rus1000_adasyn1000": {
        "nonsep_target": 1000,
        "sep_target": 1000,
        "use_rus": True,
    },

    "enn_rus500_adasyn500": {
        "nonsep_target": 500,
        "sep_target": 500,
        "use_rus": True,
    },
}


# ---------------------------------------------------------
# Create datasets in memory
# ---------------------------------------------------------
datasets_adasyn = {}

print("\nCreating ENN + ADASYN datasets:")
print(
    f"{'Dataset':<30} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Real SEP':>10} "
    f"{'Synthetic SEP':>14} "
    f"{'RUS?':>8} "
    f"{'Shape'}"
)
print("─" * 108)

for dataset_name, cfg in adasyn_configs.items():

    nonsep_target = cfg["nonsep_target"]
    sep_target = cfg["sep_target"]
    use_rus = cfg["use_rus"]

    if nonsep_target > original_nonsep:
        raise ValueError(
            f"{dataset_name}: requested {nonsep_target} Non-SEP samples, "
            f"but only {original_nonsep} are available after ENN."
        )

    if sep_target < original_sep:
        raise ValueError(
            f"{dataset_name}: requested {sep_target} SEP samples, "
            f"but original SEP count is {original_sep}. "
            f"This code keeps all real SEP samples."
        )

    n_synthetic_needed = sep_target - original_sep

    if n_synthetic_needed > len(X_synthetic_sep_all):
        raise ValueError(
            f"{dataset_name}: requested {n_synthetic_needed} synthetic SEP samples, "
            f"but only {len(X_synthetic_sep_all)} are available."
        )

    # -----------------------------------------------------
    # Select Non-SEP samples
    # -----------------------------------------------------
    if use_rus:
        selected_nonsep_idx = rng.choice(
            len(X_nonsep_all),
            size=nonsep_target,
            replace=False
        )

        X_nonsep = X_nonsep_all[selected_nonsep_idx]

    else:
        X_nonsep = X_nonsep_all.copy()

    y_nonsep = np.zeros(len(X_nonsep), dtype=np.float32)

    # -----------------------------------------------------
    # Select synthetic SEP from the fixed ADASYN pool
    # -----------------------------------------------------
    selected_syn_idx = rng.choice(
        len(X_synthetic_sep_all),
        size=n_synthetic_needed,
        replace=False
    )

    X_syn_sep = X_synthetic_sep_all[selected_syn_idx]

    # Keep all real SEP + selected synthetic SEP
    X_sep = np.concatenate([X_real_sep_all, X_syn_sep], axis=0).astype(np.float32)
    y_sep = np.ones(len(X_sep), dtype=np.float32)

    # -----------------------------------------------------
    # Combine and shuffle
    # -----------------------------------------------------
    X_res = np.concatenate([X_nonsep, X_sep], axis=0).astype(np.float32)
    y_res = np.concatenate([y_nonsep, y_sep], axis=0)

    final_perm = rng.permutation(len(y_res))
    X_res = X_res[final_perm]
    y_res = y_res[final_perm]

    datasets_adasyn[dataset_name] = {
        "X_train": X_res,
        "y_train": y_res,

        "X_val": X_val,
        "y_val": y_val,

        "X_test": X_test,
        "y_test": y_test,

        "nonsep_target": nonsep_target,
        "sep_target": sep_target,
        "use_rus": use_rus,
        "real_sep": original_sep,
        "synthetic_sep": n_synthetic_needed,
        "adasyn_infer_time": adasyn_infer_time,
        "adasyn_time_per_sample_ms": adasyn_infer_time / n_synthetic_generated * 1000,
    }

    final_counts = np.bincount(y_res.astype(int), minlength=2)

    print(
        f"{dataset_name:<30} "
        f"{final_counts[0]:>8} "
        f"{final_counts[1]:>8} "
        f"{original_sep:>10} "
        f"{n_synthetic_needed:>14} "
        f"{str(use_rus):>8} "
        f"{X_res.shape}"
    )


print("\n✅ ENN + ADASYN datasets are ready in memory.")
print("Dictionary name: datasets_adasyn")
print("Datasets:")
for key in datasets_adasyn.keys():
    print(" ", key)

print(f"\n✅ Real vs synthetic ADASYN SEP samples saved to: {os.path.abspath(SAVE_DIR)}/")
print(f"ADASYN inference/generation time       : {adasyn_infer_time:.4f}s")
print(f"ADASYN time per synthetic SEP sample   : {adasyn_infer_time / n_synthetic_generated * 1000:.4f}ms")

Loaded ENN-cleaned dataset:
Train: (12268, 288, 10) (12268,)
Val  : (1780, 288, 10) (1780,)
Test : (3559, 288, 10) (3559,)

Original ENN-cleaned training class counts:
Non-SEP: 12150
SEP    : 118
Ratio  : 103.0x
────────────────────────────────────────────────────────────────────────────────

Synthetic SEP samples to generate: 12032

Created ADASYN synthetic SEP pool:
Synthetic SEP shape           : (12049, 288, 10)
Synthetic SEP count           : 12049
ADASYN generation time        : 0.6404s
ADASYN time per synthetic SEP : 0.0531ms
────────────────────────────────────────────────────────────────────────────────
Saved real vs synthetic ADASYN SEP samples → /Users/samskanderi/SEP_DataAugmentation/sep_samples/sep_real_vs_synthetic_adasyn.pkl
Real SEP      : 118
Synthetic SEP : 118
────────────────────────────────────────────────────────────────────────────────

Creating ENN + ADASYN datasets:
Dataset                         Non-SEP      SEP   Real SEP  Synthetic SEP     RUS? Shape
──────

### Catch22 Vectorization for SVM

In [2]:
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22
from sklearn.impute import SimpleImputer
import warnings

# ══════════════════════════════════════════════════════════════
# CATCH22 FEATURE EXTRACTION FOR ENN + ADASYN DATASETS
# Input : datasets_adasyn
#         enn_adasyn_balanced,
#         enn_rus8000_adasyn8000,
#         enn_rus4000_adasyn4000,
#         enn_rus2000_adasyn2000,
#         enn_rus1000_adasyn1000,
#         enn_rus500_adasyn500
# Output: datasets_svm_adasyn
# ══════════════════════════════════════════════════════════════

def _catch22_chunk(X_chunk):
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()


def extract_catch22(X, label="", n_jobs=-1, chunk_size=200):

    n_samples, n_timepoints, n_channels = X.shape

    # Replace NaN / Inf before Catch22
    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️ {n_bad} non-finite values in {label} — replacing with channel median")

        X = X.copy()

        for c in range(n_channels):
            col = X[:, :, c]
            median_c = np.nanmedian(col)

            if not np.isfinite(median_c):
                median_c = 0.0

            col[~np.isfinite(col)] = median_c
            X[:, :, c] = col

    # sktime expects shape: (samples, channels, timesteps)
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)

    chunks = [
        X_sktime[i:i + chunk_size]
        for i in range(0, n_samples, chunk_size)
    ]

    print(f"    {label}: {n_samples} samples in {len(chunks)} chunks", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk)
        for chunk in chunks
    )

    X_feat = np.vstack(results)

    expected_cols = 22 * n_channels

    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected shape for {label}: {X_feat.shape}, "
        f"expected ({n_samples}, {expected_cols})"
    )

    return X_feat.astype(np.float32)


# ══════════════════════════════════════════════════════════════
# Sanity check before extraction
# ══════════════════════════════════════════════════════════════

print("Pre-extraction value range check:")
print(
    f"{'Dataset':<30} "
    f"{'Train shape':<22} "
    f"{'Finite?':<10} "
    f"{'Min':>12} "
    f"{'Max':>12} "
    f"{'NaN/Inf':>10} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Ratio':>8}"
)
print("─" * 135)

for dataset_name, d in datasets_adasyn.items():
    X = d["X_train"]
    y = d["y_train"]

    counts = np.bincount(y.astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    print(
        f"{dataset_name:<30} "
        f"{str(X.shape):<22} "
        f"{str(np.isfinite(X).all()):<10} "
        f"{np.nanmin(X):>12.4f} "
        f"{np.nanmax(X):>12.4f} "
        f"{(~np.isfinite(X)).sum():>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>7.1f}x"
    )


# ══════════════════════════════════════════════════════════════
# Extract Catch22 features
# ══════════════════════════════════════════════════════════════

datasets_svm_adasyn = {}

print("\nExtracting Catch22 features for ENN + ADASYN datasets...")

for dataset_name, d in datasets_adasyn.items():

    print(f"\n[{dataset_name}] extracting train...")
    X_tr = extract_catch22(
        d["X_train"],
        label=f"{dataset_name}/train",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting val...")
    X_va = extract_catch22(
        d["X_val"],
        label=f"{dataset_name}/val",
        n_jobs=-1,
        chunk_size=200
    )

    print(f"[{dataset_name}] extracting test...")
    X_te = extract_catch22(
        d["X_test"],
        label=f"{dataset_name}/test",
        n_jobs=-1,
        chunk_size=200
    )

    datasets_svm_adasyn[dataset_name] = {
        "X_train": X_tr,
        "y_train": d["y_train"],

        "X_val": X_va,
        "y_val": d["y_val"],

        "X_test": X_te,
        "y_test": d["y_test"],

        "nonsep_target": d.get("nonsep_target", None),
        "sep_target": d.get("sep_target", None),
        "use_rus": d.get("use_rus", None),

        "real_sep": d.get("real_sep", None),
        "synthetic_sep": d.get("synthetic_sep", None),
        "adasyn_infer_time": d.get("adasyn_infer_time", None),
        "adasyn_time_per_sample_ms": d.get("adasyn_time_per_sample_ms", None),
    }

    print(
        f"[{dataset_name}] ✓ "
        f"train {X_tr.shape} | "
        f"val {X_va.shape} | "
        f"test {X_te.shape}"
    )


# ══════════════════════════════════════════════════════════════
# Impute Catch22 features
# Fit imputer on train only, apply to val/test
# ══════════════════════════════════════════════════════════════

print("\nChecking and imputing Catch22 features...")

for dataset_name in list(datasets_svm_adasyn.keys()):

    d = datasets_svm_adasyn[dataset_name]

    X_tr = d["X_train"].copy()
    X_va = d["X_val"].copy()
    X_te = d["X_test"].copy()

    y_tr = d["y_train"]
    y_va = d["y_val"]
    y_te = d["y_test"]

    n_bad_tr = (~np.isfinite(X_tr)).sum()
    n_bad_va = (~np.isfinite(X_va)).sum()
    n_bad_te = (~np.isfinite(X_te)).sum()

    if n_bad_tr > 0 or n_bad_va > 0 or n_bad_te > 0:

        print(
            f"  ⚠️ [{dataset_name}] non-finite values — "
            f"train: {n_bad_tr}, val: {n_bad_va}, test: {n_bad_te} → imputing"
        )

        X_tr = np.where(np.isfinite(X_tr), X_tr, np.nan)
        X_va = np.where(np.isfinite(X_va), X_va, np.nan)
        X_te = np.where(np.isfinite(X_te), X_te, np.nan)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")

            imputer = SimpleImputer(strategy="median")

            X_tr = imputer.fit_transform(X_tr)
            X_va = imputer.transform(X_va)
            X_te = imputer.transform(X_te)

        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_va = np.nan_to_num(X_va, nan=0.0, posinf=0.0, neginf=0.0)
        X_te = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

        print(f"  ✓ [{dataset_name}] imputed")

    else:
        print(f"  ✓ [{dataset_name}] no NaN or Inf")

    datasets_svm_adasyn[dataset_name] = {
        "X_train": X_tr.astype(np.float32),
        "y_train": y_tr,

        "X_val": X_va.astype(np.float32),
        "y_val": y_va,

        "X_test": X_te.astype(np.float32),
        "y_test": y_te,

        "nonsep_target": d.get("nonsep_target", None),
        "sep_target": d.get("sep_target", None),
        "use_rus": d.get("use_rus", None),

        "real_sep": d.get("real_sep", None),
        "synthetic_sep": d.get("synthetic_sep", None),
        "adasyn_infer_time": d.get("adasyn_infer_time", None),
        "adasyn_time_per_sample_ms": d.get("adasyn_time_per_sample_ms", None),
    }


# ══════════════════════════════════════════════════════════════
# Final summary
# ══════════════════════════════════════════════════════════════

print("\n✅ SVM-ready Catch22 ENN + ADASYN datasets are ready:")
print(list(datasets_svm_adasyn.keys()))

print(
    f"\n{'Dataset':<30} "
    f"{'Train shape':<18} "
    f"{'Val shape':<18} "
    f"{'Test shape':<18} "
    f"{'Class 0':>8} "
    f"{'Class 1':>8} "
    f"{'Ratio':>8}"
)
print("─" * 120)

for name, d in datasets_svm_adasyn.items():
    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    print(
        f"{name:<30} "
        f"{str(d['X_train'].shape):<18} "
        f"{str(d['X_val'].shape):<18} "
        f"{str(d['X_test'].shape):<18} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>7.1f}x"
    )

Pre-extraction value range check:
Dataset                        Train shape            Finite?             Min          Max    NaN/Inf  Class 0  Class 1    Ratio
───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
enn_adasyn_balanced            (24300, 288, 10)       True            -6.5136      27.8326          0    12150    12150     1.0x
enn_rus8000_adasyn8000         (16000, 288, 10)       True            -6.5136      27.8326          0     8000     8000     1.0x
enn_rus4000_adasyn4000         (8000, 288, 10)        True            -6.5136      27.8326          0     4000     4000     1.0x
enn_rus2000_adasyn2000         (4000, 288, 10)        True            -6.5136      27.2851          0     2000     2000     1.0x
enn_rus1000_adasyn1000         (2000, 288, 10)        True            -6.5136      27.2851          0     1000     1000     1.0x
enn_rus500_adasyn500           (1000, 288, 10)        Tr

### Classification

In [3]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC
import time
import os

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    recall    = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    expected = ((TP + FN) * (TP + FP) + (TN + FP) * (TN + FN)) / (TP + TN + FP + FN) ** 2
    hss1     = (accuracy - expected) / (1 - expected) if (1 - expected) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'recall': recall, 'f1': f1, 'accuracy': accuracy
    }


def train_and_evaluate(model, X_train, y_train, X_test, y_test):
    t0         = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    t0         = time.time()
    y_pred     = model.predict(X_test)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test, y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics


def save_results(metrics_list, filepath):
    with open(filepath, 'w') as f:
        for m in metrics_list:
            line = (f"{m['TP']},{m['TN']},{m['FP']},{m['FN']},"
                    f"{m['tss']:.6f},{m['hss1']:.6f},{m['hss2']:.6f},{m['gss']:.6f},"
                    f"{m['recall']:.6f},{m['f1']:.6f},{m['accuracy']:.6f},"
                    f"{m['train_time']:.4f},{m['infer_time']:.4f}")
            f.write(line + "\n")


def print_results(metrics_list, title):
    keys = ['tss', 'hss1', 'hss2', 'gss', 'recall', 'f1', 'accuracy', 'train_time', 'infer_time']
    print(f"\n{'─'*55}")
    print(f"  {title}")
    print(f"{'─'*55}")
    for i, m in enumerate(metrics_list):
        print(f"  Run {i+1}: TP={m['TP']}  TN={m['TN']}  FP={m['FP']}  FN={m['FN']}")
        print(f"         TSS={m['tss']:.4f}  HSS1={m['hss1']:.4f}  HSS2={m['hss2']:.4f}  GSS={m['gss']:.4f}")
        print(f"         Recall={m['recall']:.4f}  F1={m['f1']:.4f}  Acc={m['accuracy']:.4f}")
        print(f"         Train={m['train_time']:.2f}s  Infer={m['infer_time']:.4f}s")
        print()
    print(f"  ── Average of {len(metrics_list)} runs ──")
    for k in keys:
        avg = np.mean([m[k] for m in metrics_list])
        print(f"  {k:<12} : {avg:.4f}")
    print(f"{'─'*55}")

### SVM

In [4]:
import os
import copy
import warnings
import numpy as np

from sklearn.svm import SVC
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# ══════════════════════════════════════════════════════════════
# SVM FINAL EXPERIMENTS — ENN + ADASYN
# Uses fixed best hyperparameters selected previously
# Best model: RBF-SVC, C=0.1, gamma=scale, class_weight=balanced
#
# Input datasets:
# datasets_svm_adasyn
#   enn_adasyn_balanced
#   enn_rus8000_adasyn8000
#   enn_rus4000_adasyn4000
#   enn_rus2000_adasyn2000
#   enn_rus1000_adasyn1000
#   enn_rus500_adasyn500
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best SVM hyperparameters
# ---------------------------------------------------------
best_model_class = SVC

best_params = {
    "kernel": "rbf",
    "C": 0.1,
    "gamma": "scale",
    "class_weight": "balanced",
    "cache_size": 2000
}

TARGET_VARIANTS = [
    "enn_adasyn_balanced",
    "enn_rus8000_adasyn8000",
    "enn_rus4000_adasyn4000",
    "enn_rus2000_adasyn2000",
    "enn_rus1000_adasyn1000",
    "enn_rus500_adasyn500",
]

print("═" * 70)
print("Final SVM experiments on ENN + ADASYN datasets")
print("Using fixed best hyperparameters")
print("═" * 70)
print(f"Model  : {best_model_class.__name__}")
print(f"Params : {best_params}")


# ---------------------------------------------------------
# Run final experiments on ENN + ADASYN datasets
# ---------------------------------------------------------
all_adasyn_svm_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_svm_adasyn:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_svm_adasyn[dataset_key]

    print("\n" + "─" * 70)
    print(f"Dataset : {dataset_key}")
    print(f"Model   : {best_model_class.__name__}")
    print(f"Params  : {best_params}")

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    train_ratio = train_counts[0] / train_counts[1] if train_counts[1] > 0 else np.inf

    print(f"Train shape   : {d['X_train'].shape}")
    print(f"Val shape     : {d['X_val'].shape}")
    print(f"Test shape    : {d['X_test'].shape}")
    print(f"Train class   : Non-SEP={train_counts[0]} | SEP={train_counts[1]} | Ratio={train_ratio:.1f}:1")
    print(f"Val class     : Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test class    : Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Real SEP      : {d.get('real_sep', None)}")
    print(f"Synthetic SEP : {d.get('synthetic_sep', None)}")
    print(f"Use RUS       : {d.get('use_rus', None)}")
    print(f"ADASYN time   : {d.get('adasyn_infer_time', None)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        model = best_model_class(**copy.deepcopy(best_params))

        metrics = train_and_evaluate(
            model,
            d["X_train"], d["y_train"],
            d["X_test"],  d["y_test"]
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.2f}s"
        )

    all_adasyn_svm_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"svm_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"SVM | {dataset_key}")


# ---------------------------------------------------------
# Select best ADASYN dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("SVM ENN + ADASYN summary")
print("═" * 70)

print(
    f"{'Dataset':<28} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Real SEP':>10} "
    f"{'Syn SEP':>10} "
    f"{'RUS?':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 130)

for dataset_key, metrics_list in all_adasyn_svm_results.items():

    d = datasets_svm_adasyn[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    real_sep = d.get("real_sep", None)
    synthetic_sep = d.get("synthetic_sep", None)
    use_rus = d.get("use_rus", None)

    print(
        f"{dataset_key:<28} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{str(real_sep):>10} "
        f"{str(synthetic_sep):>10} "
        f"{str(use_rus):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest SVM ENN + ADASYN dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Model   : {best_model_class.__name__}")
print(f"  Params  : {best_params}")

print("\nAll SVM ENN + ADASYN experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
Final SVM experiments on ENN + ADASYN datasets
Using fixed best hyperparameters
══════════════════════════════════════════════════════════════════════
Model  : SVC
Params : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}

──────────────────────────────────────────────────────────────────────
Dataset : enn_adasyn_balanced
Model   : SVC
Params  : {'kernel': 'rbf', 'C': 0.1, 'gamma': 'scale', 'class_weight': 'balanced', 'cache_size': 2000}
Train shape   : (24300, 220)
Val shape     : (1780, 220)
Test shape    : (3559, 220)
Train class   : Non-SEP=12150 | SEP=12150 | Ratio=1.0:1
Val class     : Non-SEP=1763 | SEP=17
Test class    : Non-SEP=3525 | SEP=34
Real SEP      : 118
Synthetic SEP : 12032
Use RUS       : False
ADASYN time   : 0.640357255935669
──────────────────────────────────────────────────────────────────────
Run 1: TSS=0.4347 | F1=0.0600 | Recall=0.6176 | Train=54

### GRU 

In [5]:
# ══════════════════════════════════════════════════════════════
# HELPER FUNCTIONS — PyTorch
# ══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import numpy as np
import time
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()
    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time
    return metrics, model


Using device: mps


In [6]:
import os
import numpy as np
import copy

# ══════════════════════════════════════════════════════════════
# GRU FINAL EXPERIMENTS — ENN + ADASYN
# Uses fixed best hyperparameters selected previously
# Best params: units=128, layers=1, dropout=0.3, lr=0.001, batch_size=32
# Datasets:
#   enn_adasyn_balanced
#   enn_rus8000_adasyn8000
#   enn_rus4000_adasyn4000
#   enn_rus2000_adasyn2000
#   enn_rus1000_adasyn1000
#   enn_rus500_adasyn500
# ══════════════════════════════════════════════════════════════

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best GRU hyperparameters
# ---------------------------------------------------------
best_params = {
    "units": 128,
    "layers": 1,
    "dropout": 0.3,
    "lr": 0.001,
    "batch_size": 32
}

TARGET_VARIANTS = [
    "enn_adasyn_balanced",
    "enn_rus8000_adasyn8000",
    "enn_rus4000_adasyn4000",
    "enn_rus2000_adasyn2000",
    "enn_rus1000_adasyn1000",
    "enn_rus500_adasyn500",
]

print(f"{'═' * 70}")
print("  Classifier : GRU (PyTorch)")
print("  Experiment : ENN + ADASYN")
print(f"  Best params: {best_params}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final GRU experiments on ADASYN datasets
# ---------------------------------------------------------
all_gru_adasyn_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_adasyn:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_adasyn[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1] if train_counts[1] > 0 else np.inf

    print("\n" + "─" * 70)
    print(f"Dataset       : {dataset_key}")
    print(f"Timesteps     : {timesteps}")
    print(f"Train         : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val           : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test          : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio         : {cr:.1f}:1")
    print(f"Real SEP      : {d.get('real_sep', None)}")
    print(f"Synthetic SEP : {d.get('synthetic_sep', None)}")
    print(f"Use RUS       : {d.get('use_rus', None)}")
    print(f"ADASYN time   : {d.get('adasyn_infer_time', None)}")
    print(f"Params        : {best_params}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_gru(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_gru_adasyn_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"gru_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"GRU | {dataset_key}")


# ---------------------------------------------------------
# Select best ADASYN dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("GRU ENN + ADASYN summary")
print("═" * 70)

print(
    f"{'Dataset':<28} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Real SEP':>10} "
    f"{'Syn SEP':>10} "
    f"{'RUS?':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)
print("─" * 135)

for dataset_key, metrics_list in all_gru_adasyn_results.items():

    d = datasets_adasyn[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    real_sep = d.get("real_sep", None)
    synthetic_sep = d.get("synthetic_sep", None)
    use_rus = d.get("use_rus", None)

    print(
        f"{dataset_key:<28} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{str(real_sep):>10} "
        f"{str(synthetic_sep):>10} "
        f"{str(use_rus):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest GRU ENN + ADASYN dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {best_params}")

print("\n✅ All GRU ENN + ADASYN experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : GRU (PyTorch)
  Experiment : ENN + ADASYN
  Best params: {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset       : enn_adasyn_balanced
Timesteps     : 288
Train         : (24300, 288, 10) | Non-SEP=12150 | SEP=12150
Val           : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test          : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio         : 1.0:1
Real SEP      : 118
Synthetic SEP : 12032
Use RUS       : False
ADASYN time   : 0.640357255935669
Params        : {'units': 128, 'layers': 1, 'dropout': 0.3, 'lr': 0.001, 'batch_size': 32}
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.1332 | F1=0.1136 | Recall=0.1471 | Train=785.0s
Run 2 …
Run 2: TSS=0.4360 | F1=0.1860 | Recall=0.4706 | Tra

In [7]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn
from transformers import PatchTSTConfig, PatchTSTForClassification

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


def build_patchtst_model(params, seq_len, n_channels):
    config = PatchTSTConfig(
        num_input_channels=n_channels,
        context_length=seq_len,
        num_targets=2,

        patch_length=params["patch_length"],
        patch_stride=params["patch_stride"],

        d_model=params["d_model"],
        num_attention_heads=params["num_attention_heads"],
        num_hidden_layers=params["num_hidden_layers"],
        ffn_dim=params["ffn_dim"],

        attention_dropout=params["dropout"],
        positional_dropout=params["dropout"],
        ff_dropout=params["dropout"],
        head_dropout=params["dropout"],

        pooling_type="mean",
        use_cls_token=True,
        norm_type="layernorm",
        channel_attention=True,
        problem_type="single_label_classification",
        scaling=None,
    )

    return PatchTSTForClassification(config)


def train_and_evaluate_patchtst(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.long).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.long).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    seq_len = X_train.shape[1]
    n_channels = X_train.shape[2]

    model = build_patchtst_model(
        params,
        seq_len=seq_len,
        n_channels=n_channels
    ).to(device)

    class_weights = torch.tensor(
        [1.0, float(class_ratio)],
        dtype=torch.float32
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            outputs = model(past_values=xb)
            logits = outputs.prediction_logits

            loss = criterion(logits, yb)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_outputs = model(past_values=X_va)
            val_logits = val_outputs.prediction_logits
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        test_outputs = model(past_values=X_te)
        test_logits = test_outputs.prediction_logits
        y_pred = torch.argmax(test_logits, dim=1).cpu().numpy()

    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [8]:
# ══════════════════════════════════════════════════════════════
# PATCHTST FINAL EXPERIMENTS — ENN + ADASYN
# Uses fixed best hyperparameters selected previously
# Best params: patch=12, stride=12, d=32, heads=4, layers=1,
#              dropout=0.2, lr=0.001, batch_size=128
# Datasets:
#   enn_adasyn_balanced
#   enn_rus8000_adasyn8000
#   enn_rus4000_adasyn4000
#   enn_rus2000_adasyn2000
#   enn_rus1000_adasyn1000
#   enn_rus500_adasyn500
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best PatchTST hyperparameters
# ---------------------------------------------------------
best_params = {
    "patch_length": 12,
    "patch_stride": 12,
    "d_model": 32,
    "num_attention_heads": 4,
    "num_hidden_layers": 1,
    "ffn_dim": 128,
    "dropout": 0.2,
    "lr": 0.001,
    "batch_size": 128,
}

TARGET_VARIANTS = [
    "enn_adasyn_balanced",
    "enn_rus8000_adasyn8000",
    "enn_rus4000_adasyn4000",
    "enn_rus2000_adasyn2000",
    "enn_rus1000_adasyn1000",
    "enn_rus500_adasyn500",
]


def patchtst_label(p):
    return (
        f"patch={p['patch_length']}  "
        f"stride={p['patch_stride']}  "
        f"d={p['d_model']}  "
        f"heads={p['num_attention_heads']}  "
        f"layers={p['num_hidden_layers']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : Hugging Face PatchTST")
print("  Experiment : ENN + ADASYN")
print(f"  Best params: {patchtst_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final PatchTST experiments on ADASYN datasets
# ---------------------------------------------------------
all_patchtst_adasyn_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_adasyn:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_adasyn[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1] if train_counts[1] > 0 else np.inf

    print("\n" + "─" * 70)
    print(f"Dataset       : {dataset_key}")
    print(f"Timesteps     : {timesteps}")
    print(f"Train         : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val           : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test          : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio         : {cr:.1f}:1")
    print(f"Real SEP      : {d.get('real_sep', None)}")
    print(f"Synthetic SEP : {d.get('synthetic_sep', None)}")
    print(f"Use RUS       : {d.get('use_rus', None)}")
    print(f"ADASYN time   : {d.get('adasyn_infer_time', None)}")
    print(f"Params        : {patchtst_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_patchtst(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_patchtst_adasyn_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"patchtst_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"PatchTST | {dataset_key}")


# ---------------------------------------------------------
# Select best ADASYN dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("PatchTST ENN + ADASYN summary")
print("═" * 70)

print(
    f"{'Dataset':<28} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Real SEP':>10} "
    f"{'Syn SEP':>10} "
    f"{'RUS?':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 135)

for dataset_key, metrics_list in all_patchtst_adasyn_results.items():

    d = datasets_adasyn[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    real_sep = d.get("real_sep", None)
    synthetic_sep = d.get("synthetic_sep", None)
    use_rus = d.get("use_rus", None)

    print(
        f"{dataset_key:<28} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{str(real_sep):>10} "
        f"{str(synthetic_sep):>10} "
        f"{str(use_rus):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest PatchTST ENN + ADASYN dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {patchtst_label(best_params)}")

print("\n✅ All PatchTST ENN + ADASYN experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : Hugging Face PatchTST
  Experiment : ENN + ADASYN
  Best params: patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset       : enn_adasyn_balanced
Timesteps     : 288
Train         : (24300, 288, 10) | Non-SEP=12150 | SEP=12150
Val           : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test          : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio         : 1.0:1
Real SEP      : 118
Synthetic SEP : 12032
Use RUS       : False
ADASYN time   : 0.640357255935669
Params        : patch=12  stride=12  d=32  heads=4  layers=1  drop=0.2  lr=0.001  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.6723 | F1=0.0941 | Recall=0.8235 | Train=76.6s
Run 2 …
Run 2: TSS=0.6431 | F1=0.0802 | Recall=0.8235 | 

In [9]:
import os
import time
import copy
import warnings
import numpy as np

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME MODEL
# ══════════════════════════════════════════════════════════════

class InceptionModule(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels=32,
        kernel_sizes=(9, 19, 39),
        bottleneck_channels=32
    ):
        super(InceptionModule, self).__init__()

        if in_channels > 1:
            self.bottleneck = nn.Conv1d(
                in_channels,
                bottleneck_channels,
                kernel_size=1,
                bias=False
            )
            conv_in_channels = bottleneck_channels
        else:
            self.bottleneck = nn.Identity()
            conv_in_channels = in_channels

        self.conv_list = nn.ModuleList([
            nn.Conv1d(
                conv_in_channels,
                out_channels,
                kernel_size=k,
                padding=k // 2,
                bias=False
            )
            for k in kernel_sizes
        ])

        self.maxpool_conv = nn.Sequential(
            nn.MaxPool1d(kernel_size=3, stride=1, padding=1),
            nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        )

        total_out_channels = out_channels * (len(kernel_sizes) + 1)

        self.bn = nn.BatchNorm1d(total_out_channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        x_bottleneck = self.bottleneck(x)

        conv_outputs = [
            conv(x_bottleneck)
            for conv in self.conv_list
        ]

        pool_output = self.maxpool_conv(x)

        out = torch.cat(conv_outputs + [pool_output], dim=1)
        out = self.bn(out)
        out = self.relu(out)

        return out


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, out_channels=32, depth=3, use_residual=True):
        super(InceptionBlock, self).__init__()

        self.use_residual = use_residual
        self.depth = depth

        modules = []
        current_channels = in_channels

        for _ in range(depth):
            module = InceptionModule(
                in_channels=current_channels,
                out_channels=out_channels
            )
            modules.append(module)

            current_channels = out_channels * 4

        self.inception_modules = nn.ModuleList(modules)

        if use_residual:
            self.residual = nn.Sequential(
                nn.Conv1d(in_channels, current_channels, kernel_size=1, bias=False),
                nn.BatchNorm1d(current_channels)
            )
            self.relu = nn.ReLU()

    def forward(self, x):
        residual = x

        out = x
        for module in self.inception_modules:
            out = module(out)

        if self.use_residual:
            residual = self.residual(residual)
            out = self.relu(out + residual)

        return out


class InceptionTimeClassifier(nn.Module):
    def __init__(self, input_channels, out_channels=32, depth=6, dropout=0.3):
        super(InceptionTimeClassifier, self).__init__()

        block_depth = max(1, depth // 2)

        self.block1 = InceptionBlock(
            in_channels=input_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        block1_channels = out_channels * 4

        self.block2 = InceptionBlock(
            in_channels=block1_channels,
            out_channels=out_channels,
            depth=block_depth,
            use_residual=True
        )

        final_channels = out_channels * 4

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(final_channels, 1)

    def forward(self, x):
        # Input shape: (batch, timesteps, features)
        # Conv1d expects: (batch, features, timesteps)
        x = x.permute(0, 2, 1)

        x = self.block1(x)
        x = self.block2(x)

        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)

        return self.fc(x).squeeze()


# ══════════════════════════════════════════════════════════════
# TRAIN AND EVALUATE INCEPTIONTIME
# ══════════════════════════════════════════════════════════════

def train_and_evaluate_inception(
    params,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    class_ratio,
    max_epochs=30,
    patience=3
):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_va = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val, dtype=torch.float32).to(device)

    X_te = torch.tensor(X_test, dtype=torch.float32).to(device)

    input_channels = X_train.shape[2]

    model = InceptionTimeClassifier(
        input_channels=input_channels,
        out_channels=params["out_channels"],
        depth=params["depth"],
        dropout=params["dropout"]
    ).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=params["lr"],
        weight_decay=1e-4
    )

    batch_size = params["batch_size"]
    n_samples = X_tr.shape[0]

    best_val_loss = float("inf")
    best_state = None
    patience_count = 0

    t0 = time.time()

    for epoch in range(max_epochs):
        model.train()

        perm = torch.randperm(n_samples, device=device)

        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]

            xb = X_tr[idx]
            yb = y_tr[idx]

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_va)
            val_loss = criterion(val_logits, y_va).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            patience_count = 0
        else:
            patience_count += 1

            if patience_count >= patience:
                break

    train_time = time.time() - t0

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    t0 = time.time()

    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()

    y_pred = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred.astype(int))
    metrics["train_time"] = train_time
    metrics["infer_time"] = infer_time

    return metrics, model

Using device: mps


In [10]:
# ══════════════════════════════════════════════════════════════
# INCEPTIONTIME FINAL EXPERIMENTS — ENN + ADASYN
# Uses fixed best hyperparameters selected previously
# Best params: channels=16, depth=6, dropout=0.3, lr=0.005, batch_size=128
# Datasets:
#   enn_adasyn_balanced
#   enn_rus8000_adasyn8000
#   enn_rus4000_adasyn4000
#   enn_rus2000_adasyn2000
#   enn_rus1000_adasyn1000
#   enn_rus500_adasyn500
# ══════════════════════════════════════════════════════════════

import os
import copy
import numpy as np

RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------------------------------------------------------
# Fixed best InceptionTime hyperparameters
# ---------------------------------------------------------
best_params = {
    "out_channels": 16,
    "depth": 6,
    "dropout": 0.3,
    "lr": 0.005,
    "batch_size": 128
}

TARGET_VARIANTS = [
    "enn_adasyn_balanced",
    "enn_rus8000_adasyn8000",
    "enn_rus4000_adasyn4000",
    "enn_rus2000_adasyn2000",
    "enn_rus1000_adasyn1000",
    "enn_rus500_adasyn500",
]


def inception_label(p):
    return (
        f"channels={p['out_channels']}  "
        f"depth={p['depth']}  "
        f"drop={p['dropout']}  "
        f"lr={p['lr']}  "
        f"bs={p['batch_size']}"
    )


print(f"{'═' * 70}")
print("  Classifier : InceptionTime")
print("  Experiment : ENN + ADASYN")
print(f"  Best params: {inception_label(best_params)}")
print(f"{'═' * 70}")


# ---------------------------------------------------------
# Run final InceptionTime experiments on ADASYN datasets
# ---------------------------------------------------------
all_inception_adasyn_results = {}

for dataset_key in TARGET_VARIANTS:

    if dataset_key not in datasets_adasyn:
        print(f"[{dataset_key}] not found — skipping")
        continue

    d = datasets_adasyn[dataset_key]

    timesteps = d["X_train"].shape[1]

    train_counts = np.bincount(d["y_train"].astype(int), minlength=2)
    val_counts   = np.bincount(d["y_val"].astype(int), minlength=2)
    test_counts  = np.bincount(d["y_test"].astype(int), minlength=2)

    cr = train_counts[0] / train_counts[1] if train_counts[1] > 0 else np.inf

    print("\n" + "─" * 70)
    print(f"Dataset       : {dataset_key}")
    print(f"Timesteps     : {timesteps}")
    print(f"Train         : {d['X_train'].shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
    print(f"Val           : {d['X_val'].shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
    print(f"Test          : {d['X_test'].shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")
    print(f"Ratio         : {cr:.1f}:1")
    print(f"Real SEP      : {d.get('real_sep', None)}")
    print(f"Synthetic SEP : {d.get('synthetic_sep', None)}")
    print(f"Use RUS       : {d.get('use_rus', None)}")
    print(f"ADASYN time   : {d.get('adasyn_infer_time', None)}")
    print(f"Params        : {inception_label(best_params)}")
    print("─" * 70)

    metrics_list = []

    for run in range(2):

        print(f"Run {run + 1} …", flush=True)

        metrics, _ = train_and_evaluate_inception(
            copy.deepcopy(best_params),
            d["X_train"], d["y_train"],
            d["X_val"],   d["y_val"],
            d["X_test"],  d["y_test"],
            class_ratio=cr,
            max_epochs=30,
            patience=3
        )

        metrics_list.append(metrics)

        precision_text = (
            f" | Precision={metrics['precision']:.4f}"
            if "precision" in metrics else ""
        )

        print(
            f"Run {run + 1}: "
            f"TSS={metrics['tss']:.4f} | "
            f"F1={metrics['f1']:.4f} | "
            f"Recall={metrics['recall']:.4f}"
            f"{precision_text} | "
            f"Train={metrics['train_time']:.1f}s"
        )

    all_inception_adasyn_results[dataset_key] = metrics_list

    filepath = os.path.join(RESULTS_DIR, f"inceptiontime_{dataset_key}.txt")
    save_results(metrics_list, filepath)
    print_results(metrics_list, f"InceptionTime | {dataset_key}")


# ---------------------------------------------------------
# Select best ADASYN dataset by average TSS
# ---------------------------------------------------------
best_dataset = None
best_avg_tss = -np.inf

print("\n" + "═" * 70)
print("InceptionTime ENN + ADASYN summary")
print("═" * 70)

print(
    f"{'Dataset':<28} "
    f"{'Train N':>10} "
    f"{'Non-SEP':>8} "
    f"{'SEP':>8} "
    f"{'Ratio':>10} "
    f"{'Real SEP':>10} "
    f"{'Syn SEP':>10} "
    f"{'RUS?':>8} "
    f"{'Avg TSS':>10} "
    f"{'Avg F1':>10} "
    f"{'Avg Recall':>12}"
)

print("─" * 135)

for dataset_key, metrics_list in all_inception_adasyn_results.items():

    d = datasets_adasyn[dataset_key]

    counts = np.bincount(d["y_train"].astype(int), minlength=2)
    ratio = counts[0] / counts[1] if counts[1] > 0 else np.inf

    avg_tss = np.mean([m["tss"] for m in metrics_list])
    avg_f1 = np.mean([m["f1"] for m in metrics_list])
    avg_recall = np.mean([m["recall"] for m in metrics_list])

    real_sep = d.get("real_sep", None)
    synthetic_sep = d.get("synthetic_sep", None)
    use_rus = d.get("use_rus", None)

    print(
        f"{dataset_key:<28} "
        f"{len(d['y_train']):>10} "
        f"{counts[0]:>8} "
        f"{counts[1]:>8} "
        f"{ratio:>9.1f}x "
        f"{str(real_sep):>10} "
        f"{str(synthetic_sep):>10} "
        f"{str(use_rus):>8} "
        f"{avg_tss:>10.4f} "
        f"{avg_f1:>10.4f} "
        f"{avg_recall:>12.4f}"
    )

    if avg_tss > best_avg_tss:
        best_avg_tss = avg_tss
        best_dataset = dataset_key


print("\nBest InceptionTime ENN + ADASYN dataset:")
print(f"  Dataset : {best_dataset}")
print(f"  Avg TSS : {best_avg_tss:.4f}")
print(f"  Params  : {inception_label(best_params)}")

print("\n✅ All InceptionTime ENN + ADASYN experiments complete.")
print(f"Results saved to: {os.path.abspath(RESULTS_DIR)}/")
print(f"Files: {sorted([f for f in os.listdir(RESULTS_DIR) if f.endswith('.txt')])}")

══════════════════════════════════════════════════════════════════════
  Classifier : InceptionTime
  Experiment : ENN + ADASYN
  Best params: channels=16  depth=6  drop=0.3  lr=0.005  bs=128
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
Dataset       : enn_adasyn_balanced
Timesteps     : 288
Train         : (24300, 288, 10) | Non-SEP=12150 | SEP=12150
Val           : (1780, 288, 10) | Non-SEP=1763 | SEP=17
Test          : (3559, 288, 10) | Non-SEP=3525 | SEP=34
Ratio         : 1.0:1
Real SEP      : 118
Synthetic SEP : 12032
Use RUS       : False
ADASYN time   : 0.640357255935669
Params        : channels=16  depth=6  drop=0.3  lr=0.005  bs=128
──────────────────────────────────────────────────────────────────────
Run 1 …
Run 1: TSS=0.0868 | F1=0.0337 | Recall=0.1765 | Train=720.0s
Run 2 …
Run 2: TSS=0.0616 | F1=0.0270 | Recall=0.1765 | Train=429.2s

─────────────────────────────────────────